# Exploring the Data

This script illustrates how to load the hdf5 files, visualize the data streams, show some summaries on the dataset, and extracts audio segments for further analysis.

In [ ]:
!pip install plotly

We begin by loading the necessary packages and setting some parameter variables.

In [ ]:
import numpy as np
import os
import h5py
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as wav_write
import plotly.graph_objects as go

# Directory path to the data
FOLDER_IN = '/home/jovyan/Data/Train/'

# Mapping of label indices to their corresponding class names
labelList = {0: 'Speech', 1: 'Cough', 2: 'Non-Verbal', 3: 'Other'}

Next, we select a particular data file and load it.

In [ ]:
# Selecting a specific file from the folder
fileList = sorted([f for f in os.listdir(FOLDER_IN) if f.endswith('.h5')])
fileName = fileList[1]

print(f"File Selected: {fileName}")

In [ ]:
# Load the selected .h5 file into a nested dictionary
file_path = os.path.join(FOLDER_IN, fileName)

def h5_to_dict(h5_obj):
    out = {}
    for key, item in h5_obj.items():
        if isinstance(item, h5py.Dataset):
            out[key] = item[()]  # NumPy array/scalar
        elif isinstance(item, h5py.Group):
            out[key] = h5_to_dict(item)  # recurse into group
    return out

with h5py.File(file_path, "r") as f:
    data = h5_to_dict(f)

print(f"Loaded: {fileName}")
print("Keys:", list(data.keys()))

Creating an interactive plot of one of the accelerometer channels. Please refresh the page if the plot does not show up in the JupyterHub. Remember to save your notebook before you refresh.

You should be able to see different levels of intensity corresponding to different activities during data collection session. We don't have the activity labels, and we will focus on the audio labels instead.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=data['acc_time'],
        y=data['acc_x'],
        mode='lines',
        name='acc_x',
        line=dict(color='steelblue')
    )
)

fig.update_layout(
    title="Interactive Plot of acc_x",
    xaxis_title="Time (s)",
    yaxis_title="Acceleration X",
    template="plotly_white",
    hovermode="x unified",
    height=450,
    width=1000
)

fig.show()

Let's take a look at the label distributions of the labels.

In [ ]:
# Getting the counts for each class label
labels = data['label']
class_ids = sorted(labelList.keys())
counts = [np.sum(labels == class_id) for class_id in class_ids]

plt.figure(figsize=(8, 4))
plt.bar(range(len(class_ids)), counts, color='steelblue')
plt.xticks(range(len(class_ids)), [labelList[class_id] for class_id in class_ids], rotation=20)
plt.ylabel("Count")
plt.title("Label Distribution")
plt.tight_layout()
plt.show()

Converting labels into a dictionary of intervals associated with each class.

In [ ]:
label_ids = sorted(np.unique(labels))
segments_by_label = {}
for i in range(0,4):
    segments_by_label[i] = []

start = 0
for i in range(1, len(labels) + 1):
    if i == len(labels) or labels[i] != labels[start]:
        lab = int(labels[start])
        segments_by_label[lab].append([int(start), int(i - 1)])
        start = i

Extracting a specific segment, plotting it and saving it for playback. You may need to refresh the page if you are trying different labels and interval IDs. Remember to save your notebook before you refresh the browser.

In [ ]:
# Selecting a label
labelID = 1 # Speech
intervalID = 0

# Converting to audio interval
intervalLabel = np.array(segments_by_label[labelID][intervalID])
intervalAudio = (intervalLabel*data['audio_sample_rate'].astype(float))/data['label_sample_rate']
intervalAudio = intervalAudio.astype(int)

# Extracting data segment
audioSegment = data['audio_data'][intervalAudio[0]:intervalAudio[1]+1]

# Plotting the segment
plt.plot(audioSegment)

# Save segment as WAV
audio_fs = int(np.asarray(data['audio_sample_rate']).squeeze())
audio_out = np.asarray(audioSegment).squeeze().astype(np.float32)
max_abs = np.max(np.abs(audio_out))
if max_abs > 0:
    audio_out = audio_out / max_abs
audio_out_int16 = (audio_out * 32767).astype(np.int16)

wav_write('audioSegment.wav', audio_fs, audio_out_int16)
print('Saved audioSegment.wav')